# Ejercicio 1 — Cobertura de Djokovic: preguntas 3 y 4

**Jugador objetivo:** Novak Djokovic  
**Fuente:** JeffSackmann/tennis_MatchChartingProject (CC BY-NC-SA 4.0)  
**Objetivo:** Calcular KPIs de cobertura para responder las preguntas 3 y 4 del P1.


In [1]:
import pandas as pd
import os

# ─── Ajusta esta ruta si es necesario ───────────────────────────────────────
DATA_DIR = "../data"
# ─────────────────────────────────────────────────────────────────────────────

MATCHES_FILE = os.path.join(DATA_DIR, "charting-m-matches.csv")
print(f"Ruta: {os.path.abspath(MATCHES_FILE)}")
print(f"Existe: {os.path.exists(MATCHES_FILE)}")

Ruta: c:\Users\pater\Desktop\DJOKOVIC\data\charting-m-matches.csv
Existe: True


In [2]:
# Carga con fallback de encoding
try:
    df = pd.read_csv(MATCHES_FILE, encoding="utf-8", low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(MATCHES_FILE, encoding="latin-1", low_memory=False)

print(f"Filas totales cargadas : {len(df):,}")
print(f"Columnas               : {list(df.columns)}")

Filas totales cargadas : 7,566
Columnas               : ['match_id', 'Player 1', 'Player 2', 'Pl 1 hand', 'Pl 2 hand', 'Date', 'Tournament', 'Round', 'Time', 'Court', 'Surface', 'Umpire', 'Best of', 'Final TB?', 'Charted by']


In [3]:
# Eliminar fila corrupta documentada en el diccionario (fila 907)
ID_CORRUPTO = "20240915-M-Davis_Cup_World_Group-RR-Botic_Van_De_Zandschulp-Matteo_Berrettini"
df = df[df["match_id"] != ID_CORRUPTO].copy()
print(f"Filas tras eliminar fila corrupta: {len(df):,}")

Filas tras eliminar fila corrupta: 7,564


In [4]:
# Filtrar partidos de Djokovic (puede ser Player 1 o Player 2)
djok = df[
    (df["Player 1"] == "Novak Djokovic") |
    (df["Player 2"] == "Novak Djokovic")
].copy()

# Columna Rival
djok["Rival"] = djok.apply(
    lambda r: r["Player 2"] if r["Player 1"] == "Novak Djokovic" else r["Player 1"],
    axis=1
)

# Columna Año (Date viene como AAAAMMDD string)
djok["Año"] = pd.to_numeric(
    djok["Date"].astype(str).str.strip().str[:4],
    errors="coerce"
).astype("Int64")

print(f"Partidos de Djokovic encontrados: {len(djok):,}")
djok[["match_id", "Player 1", "Player 2", "Rival", "Año", "Surface", "Tournament"]].head(5)

Partidos de Djokovic encontrados: 553


,match_id,Player 1,Player 2,Rival,Año,Surface,Tournament
59,20260311-M-Indian_Wells_Masters-R16-Novak_Djok...,Novak Djokovic,Jack Draper,Jack Draper,2026,Hard,Indian Wells Masters
65,20260309-M-Indian_Wells_Masters-R32-Aleksandar...,Aleksandar Kovacevic,Novak Djokovic,Aleksandar Kovacevic,2026,Hard,Indian Wells Masters
123,20260201-M-Australian_Open-F-Novak_Djokovic-Ca...,Novak Djokovic,Carlos Alcaraz,Carlos Alcaraz,2026,Hard,Australian Open
125,20260130-M-Australian_Open-SF-Jannik_Sinner-No...,Jannik Sinner,Novak Djokovic,Jannik Sinner,2026,Hard,Australian Open
127,20260128-M-Australian_Open-QF-Novak_Djokovic-L...,Novak Djokovic,Lorenzo Musetti,Lorenzo Musetti,2026,Hard,Australian Open


---
## Pregunta 3 — Distribución de partidos de Djokovic

In [5]:
# KPIs globales
n_partidos = len(djok)
n_rivales  = djok["Rival"].nunique()
n_torneos  = djok["Tournament"].nunique()
superficies = sorted(djok["Surface"].dropna().unique().tolist())

print(f"Partidos de Djokovic charteados : {n_partidos}")
print(f"Rivales únicos                  : {n_rivales}")
print(f"Torneos únicos                  : {n_torneos}")
print(f"Superficies disponibles         : {superficies}")

Partidos de Djokovic charteados : 553
Rivales únicos                  : 161
Torneos únicos                  : 51
Superficies disponibles         : ['Clay', 'Grass', 'Hard']


In [6]:
# Distribución por superficie
sup = (
    djok["Surface"]
    .value_counts()
    .rename_axis("Superficie")
    .reset_index(name="Partidos")
)
sup["% del total"] = (sup["Partidos"] / n_partidos * 100).round(1)
print("--- Distribución por SUPERFICIE ---")
print(sup.to_string(index=False))

--- Distribución por SUPERFICIE ---
Superficie  Partidos  % del total
      Hard       366         66.2
      Clay       129         23.3
     Grass        58         10.5


In [7]:
# Distribución por año (todos los años)
por_año = (
    djok.groupby("Año", dropna=True)
    .size()
    .rename("Partidos")
    .reset_index()
    .sort_values("Año")
)
print("--- Distribución por AÑO ---")
print(por_año.to_string(index=False))

--- Distribución por AÑO ---
 Año  Partidos
2005         2
2006         5
2007        22
2008        21
2009        22
2010        18
2011        25
2012        41
2013        33
2014        20
2015        51
2016        28
2017        14
2018        25
2019        28
2020        12
2021        29
2022        31
2023        45
2024        38
2025        37
2026         6


In [8]:
# Top 20 torneos
top_torneos = (
    djok["Tournament"]
    .value_counts()
    .head(20)
    .rename_axis("Torneo")
    .reset_index(name="Partidos")
)
print("--- Top 20 TORNEOS ---")
print(top_torneos.to_string(index=False))

--- Top 20 TORNEOS ---
              Torneo  Partidos
     Australian Open        62
             US Open        51
           Wimbledon        48
         Tour Finals        44
       Roland Garros        40
        Rome Masters        32
 Monte Carlo Masters        28
       Paris Masters        24
    Shanghai Masters        23
       Miami Masters        22
Indian Wells Masters        19
            Olympics        17
               Dubai        17
  Cincinnati Masters        16
      Canada Masters        14
      Madrid Masters        14
             Beijing        10
                Doha         9
              Astana         5
               Paris         5


In [9]:
# Top 20 rivales
top_rivales = (
    djok["Rival"]
    .value_counts()
    .head(20)
    .rename_axis("Rival")
    .reset_index(name="Partidos")
)
print("--- Top 20 RIVALES ---")
print(top_rivales.to_string(index=False))

--- Top 20 RIVALES ---
                Rival  Partidos
         Rafael Nadal        52
        Roger Federer        47
          Andy Murray        30
        Tomas Berdych        15
Juan Martin Del Potro        12
   Stefanos Tsitsipas        11
         Gael Monfils        11
        Stan Wawrinka        11
   Jo Wilfried Tsonga        10
        Jannik Sinner        10
      Daniil Medvedev        10
        Kei Nishikori        10
         David Ferrer        10
      Lorenzo Musetti         9
        Dominic Thiem         9
      Grigor Dimitrov         8
       Hubert Hurkacz         8
          Marin Cilic         8
      Richard Gasquet         7
       Carlos Alcaraz         7


---
## Pregunta 4 — Cobertura temporal

In [10]:
primer_año   = int(djok["Año"].min())
ultimo_año   = int(djok["Año"].max())
n_temporadas = int(djok["Año"].nunique())

print(f"Primer año con datos : {primer_año}")
print(f"Último año con datos : {ultimo_año}")
print(f"Temporadas cubiertas : {n_temporadas}  (años distintos con >= 1 partido)")
print(f"Rango total (años)   : {ultimo_año - primer_año + 1}")

# Huecos en la serie temporal
años_con_datos = set(djok["Año"].dropna().astype(int))
años_rango     = set(range(primer_año, ultimo_año + 1))
huecos         = sorted(años_rango - años_con_datos)
if huecos:
    print(f"Años sin partidos    : {huecos}")
else:
    print("Sin huecos — cobertura continua año a año")

Primer año con datos : 2005
Último año con datos : 2026
Temporadas cubiertas : 22  (años distintos con >= 1 partido)
Rango total (años)   : 22
Sin huecos — cobertura continua año a año


In [11]:
# Últimos 5 años — para razonar la ventana reciente
print(f"--- Partidos últimos 5 años ({ultimo_año - 4}–{ultimo_año}) ---")
recientes = por_año[por_año["Año"] >= ultimo_año - 4]
print(recientes.to_string(index=False))

--- Partidos últimos 5 años (2022–2026) ---
 Año  Partidos
2022        31
2023        45
2024        38
2025        37
2026         6


# NUEVOS ANALISIS

In [1]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

sd = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ServeDirection.csv", encoding="utf-8")
djoko = sd[sd['player'] == 'Novak Djokovic']

print("=== Valores únicos de 'row' ===")
print(djoko['row'].value_counts().to_string())

print(f"\nTotal filas Djokovic: {len(djoko)}")
print(f"\nEjemplo de un partido:")
mid = djoko['match_id'].iloc[0]
print(djoko[djoko['match_id'] == mid][['match_id','row','deuce_wide','deuce_t','ad_wide','ad_t']].to_string())

=== Valores únicos de 'row' ===
row
Total    550
1        550
2        550

Total filas Djokovic: 1650

Ejemplo de un partido:
                                                           match_id    row  deuce_wide  deuce_t  ad_wide  ad_t
354  20260311-M-Indian_Wells_Masters-R16-Novak_Djokovic-Jack_Draper  Total          31       14       16    28
355  20260311-M-Indian_Wells_Masters-R16-Novak_Djokovic-Jack_Draper      1          18       12       15    15
356  20260311-M-Indian_Wells_Masters-R16-Novak_Djokovic-Jack_Draper      2          13        2        1    13


In [1]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

rd = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnDepth.csv", encoding="utf-8")
djoko = rd[rd['player'] == 'Novak Djokovic']

print("=== Valores únicos de 'row' ===")
print(djoko['row'].value_counts().to_string())

=== Valores únicos de 'row' ===
row
Total    550
D        550
4A       550
6        550
v1st     550
A        550
4        550
gs       550
bh       550
fh       550
v2nd     550
4D       549
6D       549
6A       548
5        547
5D       541
5A       538
sl       520


In [2]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

rd = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnDepth.csv", encoding="utf-8")
djoko = rd[(rd['player'] == 'Novak Djokovic') & (rd['row'] == 'Total')]

total_depth = djoko['shallow'].sum() + djoko['deep'].sum() + djoko['very_deep'].sum()
total_pts = djoko['returnable'].sum() + djoko['unforced'].sum() + djoko['err_net'].sum() + djoko['err_wide'].sum() + djoko['err_wide_deep'].sum()

print("=== KPIs con denominador corregido ===")
print(f"Total puntos al resto (returnable+errores) : {total_pts:,}")
print(f"Total con profundidad clasificada          : {total_depth:,}")
print(f"returnable                                 : {djoko['returnable'].sum():,}")

print(f"\n% Restos en juego = returnable / total_pts : {djoko['returnable'].sum()/total_pts:.4f}")
print(f"% Deep+VeryDeep / total_depth              : {(djoko['deep'].sum()+djoko['very_deep'].sum())/total_depth:.4f}")
print(f"% Deep+VeryDeep / returnable               : {(djoko['deep'].sum()+djoko['very_deep'].sum())/djoko['returnable'].sum():.4f}")
print(f"% Shallow / total_depth                    : {djoko['shallow'].sum()/total_depth:.4f}")
print(f"% Very deep / total_depth                  : {djoko['very_deep'].sum()/total_depth:.4f}")

print(f"\n=== Verificar con v1st y v2nd ===")
for row_val in ['v1st', 'v2nd']:
    sub = rd[(rd['player']=='Novak Djokovic') & (rd['row']==row_val)]
    td = sub['shallow'].sum() + sub['deep'].sum() + sub['very_deep'].sum()
    print(f"\nrow = {row_val}:")
    print(f"  shallow+deep+very_deep : {td:,}")
    print(f"  returnable             : {sub['returnable'].sum():,}")
    print(f"  % deep+vd / total_depth: {(sub['deep'].sum()+sub['very_deep'].sum())/td:.4f}")

=== KPIs con denominador corregido ===
Total puntos al resto (returnable+errores) : 36,279
Total con profundidad clasificada          : 41,493
returnable                                 : 34,596

% Restos en juego = returnable / total_pts : 0.9536
% Deep+VeryDeep / total_depth              : 0.8417
% Deep+VeryDeep / returnable               : 1.0095
% Shallow / total_depth                    : 0.1583
% Very deep / total_depth                  : 0.2282

=== Verificar con v1st y v2nd ===

row = v1st:
  shallow+deep+very_deep : 24,005
  returnable             : 19,527
  % deep+vd / total_depth: 0.8157

row = v2nd:
  shallow+deep+very_deep : 17,488
  returnable             : 15,069
  % deep+vd / total_depth: 0.8775


In [3]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

ro = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
djoko = ro[(ro['player'] == 'Novak Djokovic') & (ro['row'] == 'Total')]

print("=== Totales globales ===")
for col in ['pts', 'pts_won', 'returnable', 'returnable_won', 'in_play', 'in_play_won', 'winners', 'total_shots']:
    print(f"{col:20s}: {djoko[col].sum():,}")

print(f"\n=== Verificación de identidades ===")
print(f"pts                        : {djoko['pts'].sum():,}")
print(f"returnable + (pts-returnable): {djoko['returnable'].sum():,} + {djoko['pts'].sum()-djoko['returnable'].sum():,}")
print(f"in_play <= returnable?     : {djoko['in_play'].sum() <= djoko['returnable'].sum()}")
print(f"winners <= in_play?        : {djoko['winners'].sum() <= djoko['in_play'].sum()}")

print(f"\n=== KPIs candidatos ===")
print(f"% pts_won / pts            : {djoko['pts_won'].sum()/djoko['pts'].sum():.4f}")
print(f"% returnable / pts         : {djoko['returnable'].sum()/djoko['pts'].sum():.4f}")
print(f"% returnable_won / returnable: {djoko['returnable_won'].sum()/djoko['returnable'].sum():.4f}")
print(f"% in_play / returnable     : {djoko['in_play'].sum()/djoko['returnable'].sum():.4f}")
print(f"% in_play_won / in_play    : {djoko['in_play_won'].sum()/djoko['in_play'].sum():.4f}")
print(f"% winners / in_play        : {djoko['winners'].sum()/djoko['in_play'].sum():.4f}")
print(f"avg total_shots            : {djoko['total_shots'].sum()/djoko['pts'].sum():.2f}")

print(f"\n=== Valores únicos de row ===")
print(ro[ro['player']=='Novak Djokovic']['row'].value_counts().to_string())

=== Totales globales ===
pts                 : 47,267
pts_won             : 19,128
returnable          : 34,596
returnable_won      : 17,597
in_play             : 33,475
in_play_won         : 17,597
winners             : 624
total_shots         : 227,268

=== Verificación de identidades ===
pts                        : 47,267
returnable + (pts-returnable): 34,596 + 12,671
in_play <= returnable?     : True
winners <= in_play?        : True

=== KPIs candidatos ===
% pts_won / pts            : 0.4047
% returnable / pts         : 0.7319
% returnable_won / returnable: 0.5086
% in_play / returnable     : 0.9676
% in_play_won / in_play    : 0.5257
% winners / in_play        : 0.0186
avg total_shots            : 4.81

=== Valores únicos de row ===
row
Total    550
gs       550
4A       550
6        550
4        550
A        550
v1st     550
D        550
bh       550
fh       550
v2nd     550
4D       549
6D       549
6A       548
5        547
5D       541
5A       538
89       524
7        52

In [4]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

for fname, key in [
    ("charting-m-stats-NetPoints.csv", "net"),
    ("charting-m-stats-KeyPointsServe.csv", "kps"),
    ("charting-m-stats-KeyPointsReturn.csv", "kpr")
]:
    df = pd.read_csv(f"{DATA_DIR}/{fname}", encoding="utf-8")
    djoko = df[df['player'] == 'Novak Djokovic']
    print(f"\n{'='*50}")
    print(f"=== {fname} ===")
    print(f"Valores únicos de row:")
    print(djoko['row'].value_counts().to_string())


=== charting-m-stats-NetPoints.csv ===
Valores únicos de row:
row
NetPoints           551
NetPointsRallies    551
Approach            539
ApproachRallies     536

=== charting-m-stats-KeyPointsServe.csv ===
Valores únicos de row:
row
BP        550
GP        550
Deuce     550
STotal    550

=== charting-m-stats-KeyPointsReturn.csv ===
Valores únicos de row:
row
BPO       550
GPF       550
DeuceR    550
RTotal    550


In [5]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

# NET POINTS
net = pd.read_csv(f"{DATA_DIR}/charting-m-stats-NetPoints.csv", encoding="utf-8")
djoko_net = net[net['player'] == 'Novak Djokovic']

print("=== NET POINTS ===")
for row_val in ['NetPoints', 'Approach']:
    sub = djoko_net[djoko_net['row'] == row_val]
    print(f"\nrow = {row_val}:")
    for col in ['net_pts', 'pts_won', 'net_winner', 'induced_forced', 'net_unforced', 'passed_at_net']:
        print(f"  {col:25s}: {sub[col].sum():,}")
    print(f"  % pts_won/net_pts       : {sub['pts_won'].sum()/sub['net_pts'].sum():.4f}")

# KEY POINTS SERVE
kps = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsServe.csv", encoding="utf-8")
djoko_kps = kps[kps['player'] == 'Novak Djokovic']

print("\n=== KEY POINTS SERVE ===")
for row_val in ['BP', 'GP', 'Deuce', 'STotal']:
    sub = djoko_kps[djoko_kps['row'] == row_val]
    print(f"\nrow = {row_val}:")
    for col in ['pts', 'pts_won', 'first_in', 'dfs']:
        print(f"  {col:25s}: {sub[col].sum():,}")
    print(f"  % pts_won/pts           : {sub['pts_won'].sum()/sub['pts'].sum():.4f}")

# KEY POINTS RETURN
kpr = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsReturn.csv", encoding="utf-8")
djoko_kpr = kpr[kpr['player'] == 'Novak Djokovic']

print("\n=== KEY POINTS RETURN ===")
for row_val in ['BPO', 'GPF', 'DeuceR', 'RTotal']:
    sub = djoko_kpr[djoko_kpr['row'] == row_val]
    print(f"\nrow = {row_val}:")
    for col in ['pts', 'pts_won']:
        print(f"  {col:25s}: {sub[col].sum():,}")
    print(f"  % pts_won/pts           : {sub['pts_won'].sum()/sub['pts'].sum():.4f}")

=== NET POINTS ===

row = NetPoints:
  net_pts                  : 11,137
  pts_won                  : 7,819
  net_winner               : 4,210
  induced_forced           : 2,598
  net_unforced             : 996
  passed_at_net            : 1,494
  % pts_won/net_pts       : 0.7021

row = Approach:
  net_pts                  : 8,236
  pts_won                  : 5,742
  net_winner               : 2,726
  induced_forced           : 2,168
  net_unforced             : 726
  passed_at_net            : 1,074
  % pts_won/net_pts       : 0.6972

=== KEY POINTS SERVE ===

row = BP:
  pts                      : 3,119
  pts_won                  : 1,996
  first_in                 : 1,994
  dfs                      : 95
  % pts_won/pts           : 0.6399

row = GP:
  pts                      : 9,265
  pts_won                  : 6,231
  first_in                 : 6,072
  dfs                      : 265
  % pts_won/pts           : 0.6725

row = Deuce:
  pts                      : 4,956
  pts_won        

In [6]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"
sb = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ServeBasics.csv", encoding="utf-8")
djoko = sb[sb['player'] == 'Novak Djokovic']

for row_val in ['1', '2']:
    sub = djoko[djoko['row'] == row_val]
    total = sub['wide'].sum() + sub['body'].sum() + sub['t'].sum()
    print(f"\nrow = {row_val}:")
    print(f"  % Wide : {sub['wide'].sum()/total:.4f}")
    print(f"  % Body : {sub['body'].sum()/total:.4f}")
    print(f"  % T    : {sub['t'].sum()/total:.4f}")
    print(f"  Suma   : {(sub['wide'].sum()+sub['body'].sum()+sub['t'].sum())/total:.4f}")


row = 1:
  % Wide : 0.4790
  % Body : 0.1037
  % T    : 0.4172
  Suma   : 1.0000

row = 2:
  % Wide : 0.3758
  % Body : 0.3272
  % T    : 0.2970
  Suma   : 1.0000


In [1]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

ro = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
djoko = ro[ro['player'] == 'Novak Djokovic']

print("=== Valores numéricos de row (buckets de rally) ===")
numeric_rows = djoko[djoko['row'].str.match(r'^\d+$', na=False)]
print(numeric_rows['row'].value_counts().sort_index().to_string())

=== Valores numéricos de row (buckets de rally) ===
row
4     550
5     547
6     550
7     524
89    524
9     520


In [2]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

ro = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
djoko = ro[ro['player'] == 'Novak Djokovic']

print("=== Buckets numéricos — totales ===")
numeric_rows = djoko[djoko['row'].str.match(r'^\d+$', na=False)]
for row_val in sorted(numeric_rows['row'].unique(), key=lambda x: int(x)):
    sub = djoko[djoko['row'] == row_val]
    print(f"\nrow = {row_val}:")
    print(f"  pts        : {sub['pts'].sum():,}")
    print(f"  pts_won    : {sub['pts_won'].sum():,}")
    print(f"  % pts_won  : {sub['pts_won'].sum()/sub['pts'].sum():.4f}")

# También serve_basics para rallies cortos
sb = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ServeBasics.csv", encoding="utf-8")
djoko_sb = sb[(sb['player'] == 'Novak Djokovic') & (sb['row'] == 'Total')]
total_pts = djoko_sb['pts'].sum()
lte3 = djoko_sb['pts_won_lte_3_shots'].sum()
print(f"\n=== Serve Basics — rallies cortos (≤3 golpes) ===")
print(f"  pts_won_lte_3_shots : {lte3:,}")
print(f"  total pts al saque  : {total_pts:,}")
print(f"  % pts_won_lte_3     : {lte3/total_pts:.4f}")

=== Buckets numéricos — totales ===

row = 4:
  pts        : 18,621
  pts_won    : 6,692
  % pts_won  : 0.3594

row = 5:
  pts        : 11,038
  pts_won    : 5,342
  % pts_won  : 0.4840

row = 6:
  pts        : 16,062
  pts_won    : 5,557
  % pts_won  : 0.3460

row = 7:
  pts        : 6,567
  pts_won    : 2,954
  % pts_won  : 0.4498

row = 9:
  pts        : 9,467
  pts_won    : 5,575
  % pts_won  : 0.5889

row = 89:
  pts        : 25,459
  pts_won    : 13,781
  % pts_won  : 0.5413

=== Serve Basics — rallies cortos (≤3 golpes) ===
  pts_won_lte_3_shots : 15,441
  total pts al saque  : 45,548
  % pts_won_lte_3     : 0.3390


In [3]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

# Verificar qué representa cada bucket leyendo el MatchChart.md
# y cruzando con los totales de overview para ver si los pts suman

ro = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
djoko = ro[ro['player'] == 'Novak Djokovic']

# Sumar todos los buckets numéricos
numeric_rows = djoko[djoko['row'].str.match(r'^\d+$', na=False)]
total_pts_buckets = numeric_rows.groupby('match_id')['pts'].sum().sum()

# Comparar con Total
total_row = djoko[djoko['row'] == 'Total']['pts'].sum()

print(f"Suma pts en buckets numéricos : {total_pts_buckets:,}")
print(f"pts en row=Total              : {total_row:,}")
print(f"Diferencia                    : {total_pts_buckets - total_row:,}")

# Ver si 4+5+6+7+9+89 se solapan
print(f"\nSuma simple de pts por bucket:")
for row_val in ['4','5','6','7','9','89']:
    sub = djoko[djoko['row'] == row_val]
    print(f"  row={row_val}: {sub['pts'].sum():,}")

suma = sum(djoko[djoko['row']==r]['pts'].sum() for r in ['4','5','6','7','9','89'])
print(f"\nSuma total buckets : {suma:,}")
print(f"row=Total          : {total_row:,}")
print(f"ratio              : {suma/total_row:.4f}")

Suma pts en buckets numéricos : 87,214
pts en row=Total              : 47,267
Diferencia                    : 39,947

Suma simple de pts por bucket:
  row=4: 18,621
  row=5: 11,038
  row=6: 16,062
  row=7: 6,567
  row=9: 9,467
  row=89: 25,459

Suma total buckets : 87,214
row=Total          : 47,267
ratio              : 1.8451


In [1]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

matches = pd.read_csv(f"{DATA_DIR}/charting-m-matches.csv", encoding="utf-8")
overview = pd.read_csv(f"{DATA_DIR}/charting-m-stats-Overview.csv", encoding="utf-8")
net = pd.read_csv(f"{DATA_DIR}/charting-m-stats-NetPoints.csv", encoding="utf-8")

# Filtrar superficie válida
valid_surfaces = ['Clay', 'Grass', 'Hard']
matches = matches[matches['Surface'].isin(valid_surfaces)]

# Djokovic en overview
djoko_ov = overview[(overview['player']=='Novak Djokovic') & (overview['set']=='Total')]
djoko_ov = djoko_ov.merge(matches[['match_id','Surface']], on='match_id', how='inner')

print("=== KPIs por superficie ===")
for surf in ['Hard', 'Clay', 'Grass']:
    sub = djoko_ov[djoko_ov['Surface']==surf]
    pts_saque = sub['first_won'].sum() + sub['second_won'].sum()
    print(f"\n{surf} ({len(sub)} partidos):")
    print(f"  % Pts Al Saque  : {pts_saque/sub['serve_pts'].sum():.4f}")
    print(f"  % Pts Al Resto  : {sub['return_pts_won'].sum()/sub['return_pts'].sum():.4f}")
    print(f"  % 1S Dentro     : {sub['first_in'].sum()/sub['serve_pts'].sum():.4f}")
    print(f"  % BP Salvados   : {sub['bp_saved'].sum()/sub['bk_pts'].sum():.4f}")

# Net points por superficie
djoko_net = net[(net['player']=='Novak Djokovic') & (net['row']=='NetPoints')]
djoko_net = djoko_net.merge(matches[['match_id','Surface']], on='match_id', how='inner')

print("\n=== Red por superficie ===")
for surf in ['Hard', 'Clay', 'Grass']:
    sub = djoko_net[djoko_net['Surface']==surf]
    if len(sub) > 0:
        print(f"\n{surf}:")
        print(f"  % Pts En Red    : {sub['pts_won'].sum()/sub['net_pts'].sum():.4f}")
        print(f"  net_pts totales : {sub['net_pts'].sum():,}")

=== KPIs por superficie ===

Hard (364 partidos):
  % Pts Al Saque  : 0.6754
  % Pts Al Resto  : 0.4083
  % 1S Dentro     : 0.6475
  % BP Salvados   : 0.6453

Clay (128 partidos):
  % Pts Al Saque  : 0.6409
  % Pts Al Resto  : 0.4110
  % 1S Dentro     : 0.6566
  % BP Salvados   : 0.6101

Grass (58 partidos):
  % Pts Al Saque  : 0.6883
  % Pts Al Resto  : 0.3768
  % 1S Dentro     : 0.6699
  % BP Salvados   : 0.6845

=== Red por superficie ===

Hard:
  % Pts En Red    : 0.7016
  net_pts totales : 6,874

Clay:
  % Pts En Red    : 0.6928
  net_pts totales : 2,702

Grass:
  % Pts En Red    : 0.7201
  net_pts totales : 1,561


In [1]:
import pandas as pd

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

matches = pd.read_csv(f"{DATA_DIR}/charting-m-matches.csv", encoding="utf-8")
overview = pd.read_csv(f"{DATA_DIR}/charting-m-stats-Overview.csv", encoding="utf-8")
resultados = pd.read_csv(f"{DATA_DIR}/djokovic_resultados.csv", encoding="utf-8")
net = pd.read_csv(f"{DATA_DIR}/charting-m-stats-NetPoints.csv", encoding="utf-8")
kps = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsServe.csv", encoding="utf-8")
kpr = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsReturn.csv", encoding="utf-8")

# Merge con resultado
djoko_ov = overview[(overview['player']=='Novak Djokovic') & (overview['set']=='Total')]
djoko_ov = djoko_ov.merge(resultados, on='match_id', how='inner')
djoko_ov = djoko_ov[djoko_ov['Resultado'].isin(['W','L'])]

print("=== Overview W vs L ===")
for res in ['W', 'L']:
    sub = djoko_ov[djoko_ov['Resultado']==res]
    pts_saque = sub['first_won'].sum() + sub['second_won'].sum()
    print(f"\n{res} ({len(sub)} partidos):")
    print(f"  % Pts Al Saque  : {pts_saque/sub['serve_pts'].sum():.4f}")
    print(f"  % Pts Al Resto  : {sub['return_pts_won'].sum()/sub['return_pts'].sum():.4f}")
    print(f"  % 1S Dentro     : {sub['first_in'].sum()/sub['serve_pts'].sum():.4f}")
    print(f"  % BP Salvados   : {sub['bp_saved'].sum()/sub['bk_pts'].sum():.4f}")

# KP
djoko_kps = kps[kps['player']=='Novak Djokovic'].merge(resultados, on='match_id', how='inner')
djoko_kpr = kpr[kpr['player']=='Novak Djokovic'].merge(resultados, on='match_id', how='inner')

print("\n=== Puntos Clave W vs L ===")
for res in ['W', 'L']:
    bp_s = djoko_kps[(djoko_kps['Resultado']==res) & (djoko_kps['row']=='BP')]
    bp_r = djoko_kpr[(djoko_kpr['Resultado']==res) & (djoko_kpr['row']=='BPO')]
    print(f"\n{res}:")
    print(f"  % BP Salvados   : {bp_s['pts_won'].sum()/bp_s['pts'].sum():.4f}")
    print(f"  % BP Convertidos: {bp_r['pts_won'].sum()/bp_r['pts'].sum():.4f}")

# Net
djoko_net = net[(net['player']=='Novak Djokovic') & (net['row']=='NetPoints')]
djoko_net = djoko_net.merge(resultados, on='match_id', how='inner')
djoko_net = djoko_net[djoko_net['Resultado'].isin(['W','L'])]

print("\n=== Red W vs L ===")
for res in ['W', 'L']:
    sub = djoko_net[djoko_net['Resultado']==res]
    print(f"\n{res}:")
    print(f"  % Pts En Red    : {sub['pts_won'].sum()/sub['net_pts'].sum():.4f}")

=== Overview W vs L ===

W (421 partidos):
  % Pts Al Saque  : 0.6937
  % Pts Al Resto  : 0.4281
  % 1S Dentro     : 0.6543
  % BP Salvados   : 0.6824

L (121 partidos):
  % Pts Al Saque  : 0.5916
  % Pts Al Resto  : 0.3262
  % 1S Dentro     : 0.6480
  % BP Salvados   : 0.5684

=== Puntos Clave W vs L ===

W:
  % BP Salvados   : 0.6824
  % BP Convertidos: 0.4468

L:
  % BP Salvados   : 0.5684
  % BP Convertidos: 0.3156

=== Red W vs L ===

W:
  % Pts En Red    : 0.7291

L:
  % Pts En Red    : 0.6216
